# Zero-shot detectors (role C): ppl + logrank
No logic here: every cell just calls `src/zeroshot.py`. Settings: GPU T4, Internet on, dataset `team20-m4-processed` attached.

**What this step does:** downloads the project code from GitHub, installs the Python packages, links the attached dataset into `data/processed`, and checks that every id in the split files exists in the data. It is needed because a fresh Kaggle session starts empty.

**What you should see:** the `ls` listing shows `m4_en.parquet`, and the split check finishes without an error.

**How long:** quick, but it depends on the network (clone and pip install need Internet). No measured time is recorded for this step.

In [ ]:
!git clone -b main https://github.com/NIRMAL20062/generated_text_detection.git
%cd generated_text_detection
# If pip breaks torch, comment this line out: Kaggle's preinstalled packages are enough.
!pip install -q -r requirements.txt
# The dataset mounts under /kaggle/input/... (path differs by account), so find it.
!mkdir -p data && ln -sfn "$(dirname "$(find /kaggle/input -name m4_en.parquet | head -1)")" data/processed
!ls -l data/processed/   # must show m4_en.parquet
!python -m src.data --check-splits


**What this step does:** scores every test text with the perplexity detector (`ppl`). A language model reads each text; a low perplexity means the text looks machine-written. It scores all settings (in-distribution, leave-one-generator-out, paraphrase, cross-domain) in one go and writes `results/ppl/<setting>/preds.csv`.

**What you should see:** progress output, then one result folder per setting under `results/ppl`.

**How long:** about 11 minutes on a Kaggle T4 GPU (640.6 s for the whole run, from `results/ppl/indist/run_info.json`).

In [ ]:
!python -m src.zeroshot --method ppl

**What this step does:** same as the perplexity step, but the score is the average log-rank of each word (how highly the language model ranked the word that was actually written). Machine text uses more top-ranked words. Writes `results/logrank/<setting>/preds.csv`.

**What you should see:** progress output, then one result folder per setting under `results/logrank`.

**How long:** about 11 minutes on a Kaggle T4 GPU (645.1 s for the whole run, from `results/logrank/indist/run_info.json`).

In [ ]:
!python -m src.zeroshot --method logrank

**What this step does:** runs Fast-DetectGPT, the fallback for DetectGPT. It compares how likely the text is under the model with how likely typical alternative texts would be. By default it scores only the DetectGPT subset and the paraphrase test ids (add `--full` to score everything), which is why it is much shorter.

**What you should see:** progress output, then folders under `results/fastdetectgpt`.

**How long:** about 1 minute on a Kaggle T4 GPU (50.5 s, from `results/fastdetectgpt/indist/run_info.json`).

In [ ]:
# Fast-DetectGPT (fallback for DetectGPT): DetectGPT subset + paraphrase test ids only. Add --full to score everything.
!python -m src.zeroshot --method fastdetectgpt

**What this step does:** computes the metrics (for example AUROC) for every `preds.csv` with the shared evaluator `src/eval.py`, and prints one summary line per setting. If `src/eval.py` is missing on this branch, it downloads just that file first.

**What you should see:** one summary line per setting, and no line starting with `FAILED`.

**How long:** quick (no measured time; it only reads the saved prediction files, no GPU is needed).

In [ ]:
# Metrics for every setting with the shared evaluator.
# src/eval.py is only on the a-data branch until PR #19 merges, so fetch just that file if it is missing.
import glob, os, subprocess, sys, urllib.request
EVAL_URL = "https://raw.githubusercontent.com/NIRMAL20062/generated_text_detection/a-data/src/eval.py"
if not os.path.exists("src/eval.py"):
    urllib.request.urlretrieve(EVAL_URL, "src/eval.py")
for method in ("ppl", "logrank", "fastdetectgpt"):
    for p in sorted(glob.glob(f"results/{method}/*/preds.csv")):
        r = subprocess.run([sys.executable, "-m", "src.eval", p], capture_output=True, text=True)
        print((r.stdout.strip().splitlines() or [""])[0] if r.returncode == 0 else f"FAILED {p}: {r.stderr.strip().splitlines()[-1]}")


**What this step does:** lists the result folders of the three zero-shot methods so you can confirm they were written.

**What you should see:** `results/ppl`, `results/logrank` and `results/fastdetectgpt`, each with one folder per setting.

**How long:** instant.

In [ ]:
!ls results/ppl results/logrank results/fastdetectgpt
